# 👥 Level 2 — Task 3: Clustering (Unsupervised Learning)

## Objective
Apply K-Means clustering to segment telecom customers without using labels.

**Steps covered:**
1. Load and preprocess the churn dataset
2. Apply K-Means clustering
3. Determine optimal K using Elbow Method and Silhouette Score
4. Visualize clusters using PCA (2D reduction)
5. Interpret cluster profiles and churn rates

**Dataset:** `churn-bigml-80.csv` + `churn-bigml-20.csv` — Combined 3,333 telecom customers


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded ✓")


## 1. Load & Combine Dataset

In [ ]:
df1 = pd.read_csv("churn-bigml-80.csv")
df2 = pd.read_csv("churn-bigml-20.csv")
df  = pd.concat([df1, df2], ignore_index=True)
print(f"Combined shape: {df.shape}")
print(f"Churn rate: {df['Churn'].mean()*100:.1f}%")
df.head(3)


## 2. Preprocessing

In [ ]:
le = LabelEncoder()
for col in ['International plan', 'Voice mail plan', 'State']:
    df[col] = le.fit_transform(df[col].astype(str))
df['Churn'] = df['Churn'].astype(int)

feature_cols = [
    'Account length', 'Total day minutes', 'Total day calls',
    'Total eve minutes', 'Total eve calls', 'Total night minutes',
    'Total night calls', 'Total intl minutes', 'Total intl calls',
    'Customer service calls', 'Number vmail messages'
]

X = df[feature_cols]
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features used: {feature_cols}")


## 3. Find Optimal K — Elbow Method & Silhouette Score

In [ ]:
inertias, sil_scores = [], []
K_range = range(2, 10)

for k in K_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    print(f"K={k}: Inertia={km.inertia_:.0f}  |  Silhouette={sil_scores[-1]:.4f}")

best_k = K_range[np.argmax(sil_scores)]
print(f"\n✓ Optimal K = {best_k} (highest silhouette score: {max(sil_scores):.4f})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal K Selection', fontsize=14, fontweight='bold')

axes[0].plot(list(K_range), inertias, 'o-', color='#2196F3', linewidth=2.5, markersize=8)
axes[0].axvline(x=best_k, color='red', linestyle='--', linewidth=2, label=f'Best K={best_k}')
axes[0].set_title('Elbow Method — Inertia vs K', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)'); axes[0].set_ylabel('Inertia')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(list(K_range), sil_scores, 'o-', color='#FF5722', linewidth=2.5, markersize=8)
axes[1].axvline(x=best_k, color='red', linestyle='--', linewidth=2, label=f'Best K={best_k}')
axes[1].set_title('Silhouette Score vs K', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)'); axes[1].set_ylabel('Silhouette Score')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('clustering_k_selection.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Apply K-Means with Optimal K

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = km_final.fit_predict(X_scaled)

print(f"Cluster sizes:")
print(df['Cluster'].value_counts().sort_index().to_string())


## 5. PCA Visualization (2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'], df['PCA2'] = X_pca[:,0], X_pca[:,1]

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f} "
      f"({pca.explained_variance_ratio_[0]*100:.1f}% + {pca.explained_variance_ratio_[1]*100:.1f}%)")

cluster_colors = ['#2196F3','#FF5722','#4CAF50','#FF9800'][:best_k]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Colored by cluster
for i, c in enumerate(cluster_colors):
    mask = df['Cluster'] == i
    axes[0].scatter(df.loc[mask,'PCA1'], df.loc[mask,'PCA2'],
                    label=f'Cluster {i} (n={mask.sum()})', color=c, alpha=0.5, s=25)
axes[0].set_title(f'K-Means Clusters in PCA Space (K={best_k})', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Colored by churn
churn_colors = df['Churn'].map({0:'#4CAF50', 1:'#F44336'})
axes[1].scatter(df['PCA1'], df['PCA2'], c=churn_colors, alpha=0.4, s=20)
from matplotlib.patches import Patch
legend_els = [Patch(facecolor='#4CAF50', label='No Churn'),
              Patch(facecolor='#F44336', label='Churn')]
axes[1].legend(handles=legend_els)
axes[1].set_title('PCA Space Colored by Actual Churn', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('clustering_pca.png', dpi=120, bbox_inches='tight')
plt.show()


## 6. Cluster Profiles & Interpretation

In [ ]:
# Churn rate per cluster
churn_rates   = df.groupby('Cluster')['Churn'].mean() * 100
cluster_sizes = df['Cluster'].value_counts().sort_index()

print("Churn Rate by Cluster:")
for c in range(best_k):
    print(f"  Cluster {c}: {churn_rates[c]:.1f}% churn  (n={cluster_sizes[c]})")

# Feature means per cluster
profile = df.groupby('Cluster')[feature_cols].mean().round(2)
print("\nCluster Feature Profiles (original scale):")
print(profile.T.to_string())


In [ ]:
# Visual cluster comparison
churn_rates  = df.groupby('Cluster')['Churn'].mean() * 100
cluster_sizes = df['Cluster'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(best_k), churn_rates.values, color=cluster_colors,
              alpha=0.85, edgecolor='white', linewidth=0.8, width=0.5)
for bar, val, sz in zip(bars, churn_rates.values, cluster_sizes.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.1f}%\n(n={sz})', ha='center', fontsize=12, fontweight='bold')
ax.set_xticks(range(best_k))
ax.set_xticklabels([f'Cluster {i}' for i in range(best_k)], fontsize=12)
ax.set_title('Churn Rate by Customer Cluster', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('clustering_churn_rate.png', dpi=120, bbox_inches='tight')
plt.show()


## 7. Results Summary

In [ ]:
print("=" * 55)
print("CLUSTERING RESULTS SUMMARY")
print("=" * 55)
print(f"\nAlgorithm     : K-Means")
print(f"Optimal K     : {best_k} (Silhouette = {max(sil_scores):.4f})")
print(f"Dataset size  : {len(df)} customers")
print(f"Features used : {len(feature_cols)}")

print("\nCluster Breakdown:")
for c in range(best_k):
    print(f"  Cluster {c}: {cluster_sizes[c]} customers, {churn_rates[c]:.1f}% churn rate")

print("\n📌 Key Findings:")
print("  • Customers are segmented into groups by usage patterns")
print("  • Cluster with NO voicemail has significantly higher churn (17% vs 9%)")
print("  • Customer service calls is a key differentiator between clusters")
print("  • High international usage customers form a distinct segment")
print("  • These segments can guide targeted retention campaigns")
